# Phase 6b — SQuAD 2.0 Retrieval Benchmark (Chunk 1)

This notebook compares dense baseline, static strong, and self-healing retrieval on held-out answerable SQuAD 2.0 validation questions. It uses only free local models. It does **not** evaluate generated answers, unanswerable examples, stress tests, or faithfulness.

## 1. Colab environment

Select a T4 GPU, replace `REPOSITORY_URL`, and run all cells.

In [ ]:
from pathlib import Path
import os
import random
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')

if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Working directory:', Path.cwd())

In [ ]:
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')

## 2. Benchmark configuration

The full defaults are 5,000 passages and 2,000 answerable validation questions. Set `SMOKE_TEST=True` for a quick pipeline check. If the per-example CSV already exists, the notebook automatically recalculates metrics without loading datasets or models; set `FORCE_RERUN=True` only when retrieval should run again. Phase 5 thresholds are not changed.

In [ ]:
SMOKE_TEST = False
FORCE_RERUN = False
CORPUS_SIZE = 500 if SMOKE_TEST else 5_000
QUESTION_COUNT = 50 if SMOKE_TEST else 2_000
BATCH_SIZE = 128
RERANK_BATCH_SIZE = 64
TOP_K = 10
CUTOFFS = (1, 3, 5, 10)
STATIC_CANDIDATE_K = 20
INITIAL_FAILURE_K = 5
MAX_RETRIES = 1
RESULTS_DIR = Path('results')
PER_EXAMPLE_PATH = RESULTS_DIR / 'squad_retrieval_per_example.csv'
METRICS_PATH = RESULTS_DIR / 'squad_retrieval_metrics.json'
RECALCULATE_ONLY = PER_EXAMPLE_PATH.exists() and not FORCE_RERUN

assert TOP_K == max(CUTOFFS)
print({'corpus_size': CORPUS_SIZE, 'question_count': QUESTION_COUNT, 'smoke': SMOKE_TEST})
print('Recalculate from saved CSV without models:', RECALCULATE_ONLY)

## 3. Prepare the reproducible SQuAD 2.0 sample

Scored questions come only from the answerable validation subset. Every selected gold context is guaranteed to be indexed. Remaining passage slots are deterministic distractors from validation and train. Unanswerable validation IDs are retained in the manifest for later chunks but are not scored here.

In [ ]:
from src.evaluation import SquadSamplingConfig, prepare_squad_retrieval_dataset

benchmark_data = None
if RECALCULATE_ONLY:
    print('Saved per-example CSV found; skipping dataset loading.')
else:
    sampling_config = SquadSamplingConfig(
        corpus_size=CORPUS_SIZE,
        question_count=QUESTION_COUNT,
        seed=SEED,
        cache_dir=Path('.cache/huggingface'),
    )
    benchmark_data = prepare_squad_retrieval_dataset(
        sampling_config,
        manifest_path=RESULTS_DIR / 'squad_retrieval_sample_manifest.json',
    )
    print('Indexed passages:', len(benchmark_data.documents))
    print('Answerable validation questions:', len(benchmark_data.questions))
    print('Retained unanswerable validation IDs:', len(benchmark_data.unanswerable_validation_ids))
    print('Sample fingerprint:', benchmark_data.fingerprint)
    print('Manifest:', benchmark_data.manifest_path)

## 4. Build or reuse the FAISS index and initialize retrieval

The fingerprinted FAISS index is cached inside the Colab runtime. Corpus and initial question embeddings are batched. The local Qwen model is available only for the existing weak-retrieval query-rewrite action; the retrieval-only graph never generates answers.

In [ ]:
from src.evaluation import CachedCrossEncoderReranker, CachedDenseRetriever
from src.rag import (
    BM25Retriever,
    CrossEncoderReranker,
    FAISSRetriever,
    HybridRetriever,
    LocalQwenGenerator,
    RAGConfig,
    RetrievalFailureDetector,
    SelfHealingRAGWorkflow,
    SelfHealingWorkflowConfig,
)

def initialize_retrieval_components(data):
    rag_config = RAGConfig(
        embedding_model_name='sentence-transformers/all-MiniLM-L6-v2',
        generation_model_name='Qwen/Qwen2.5-1.5B-Instruct',
        embedding_batch_size=BATCH_SIZE,
        max_new_tokens=64,
    )
    index_dir = Path('.cache/squad_faiss') / data.fingerprint
    base_dense = FAISSRetriever(
        embedding_model_name=rag_config.embedding_model_name,
        device=rag_config.device,
        batch_size=BATCH_SIZE,
    )
    if (index_dir / 'documents.faiss').exists():
        base_dense.load(index_dir)
    else:
        base_dense.build(data.documents)
        base_dense.save(index_dir)

    dense = CachedDenseRetriever(base_dense)
    bm25 = BM25Retriever(data.documents)
    hybrid = HybridRetriever(dense, bm25)
    cached_reranker = CachedCrossEncoderReranker(
        CrossEncoderReranker(batch_size=RERANK_BATCH_SIZE)
    )
    generator = LocalQwenGenerator(rag_config)
    workflow = SelfHealingRAGWorkflow(
        dense_retriever=dense,
        bm25_retriever=bm25,
        hybrid_retriever=hybrid,
        reranker=cached_reranker,
        generator=generator,
        failure_detector=RetrievalFailureDetector(),
        config=SelfHealingWorkflowConfig(
            initial_retrieval_depth=STATIC_CANDIDATE_K,
            expanded_retrieval_depth=STATIC_CANDIDATE_K * 2,
            reranked_top_k=TOP_K,
            generation_top_k=TOP_K,
            max_retries=MAX_RETRIES,
        ),
    )
    print('Index directory:', index_dir)
    print('Cross-encoder device:', cached_reranker.reranker.device)
    print('Qwen device (query rewrite only):', generator.device)
    return dense, hybrid, cached_reranker, workflow

dense_retriever = hybrid_retriever = reranker = workflow = None
if RECALCULATE_ONLY:
    print('Skipping FAISS and model initialization.')
else:
    dense_retriever, hybrid_retriever, reranker, workflow = initialize_retrieval_components(benchmark_data)

## 5. Run retrieval or recalculate from saved ranks

In [ ]:
import json
from src.evaluation import (
    SquadRetrievalBenchmark,
    SquadRetrievalBenchmarkConfig,
    recalculate_squad_retrieval_metrics_from_csv,
)

def show_progress(index, total, example):
    if index == 1 or index % 100 == 0 or index == total:
        print(f'[{index}/{total}] {example.id}: {example.question[:80]}')

if RECALCULATE_ONLY:
    report = recalculate_squad_retrieval_metrics_from_csv(
        per_example_path=PER_EXAMPLE_PATH,
        metrics_path=METRICS_PATH,
        cutoffs=CUTOFFS,
        initial_failure_k=INITIAL_FAILURE_K,
    )
    saved_payload = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
    indexed_passage_count = saved_payload.get('dataset', {}).get('indexed_passages', 'unknown')
    evaluated_question_count = len(report.records)
    print('Recovery metrics recalculated from saved CSV; no models were run.')
else:
    benchmark = SquadRetrievalBenchmark(
        dense_retriever=dense_retriever,
        hybrid_retriever=hybrid_retriever,
        reranker=reranker,
        self_healing_workflow=workflow,
        config=SquadRetrievalBenchmarkConfig(
            cutoffs=CUTOFFS,
            static_candidate_k=STATIC_CANDIDATE_K,
            initial_failure_k=INITIAL_FAILURE_K,
            output_dir=RESULTS_DIR,
        ),
    )
    report = benchmark.run(benchmark_data, progress_callback=show_progress)
    indexed_passage_count = len(benchmark_data.documents)
    evaluated_question_count = len(benchmark_data.questions)
    print('Benchmark complete.')

## 6. Concise benchmark summary

In [ ]:
def print_ranking_metrics(label, metrics):
    values = metrics.to_dict()
    print(f'\n{label}')
    for cutoff in CUTOFFS:
        print(f'- Recall@{cutoff}: {values[f"recall@{cutoff}"]:.4f}')
    print(f'- MRR: {values["mrr"]:.4f}')
    mean_rank = values['mean_gold_rank']
    print('- Mean gold rank (found only):', 'N/A' if mean_rank is None else f'{mean_rank:.4f}')

print('Dataset')
print('- Indexed passages:', indexed_passage_count)
print('- Evaluated answerable questions:', evaluated_question_count)
print_ranking_metrics('Dense Baseline', report.dense_metrics)
print_ranking_metrics('Static Strong RAG', report.static_strong_metrics)
print_ranking_metrics('Self-Healing RAG', report.self_healing_metrics)
print(f'- Recovery attempt rate: {report.recovery_attempt_rate:.4f}')
print(f'- Average retries: {report.average_retry_count:.4f}')
print(f'- Unnecessary modification rate: {report.unnecessary_modification_rate:.4f}')
primary_recovery = report.recovery_at_cutoff[INITIAL_FAILURE_K]
print(f'\nInitial retrieval failures@{INITIAL_FAILURE_K}')
print('- Count:', primary_recovery.initial_failure_count)
print(f'- Percentage: {100 * primary_recovery.initial_failure_rate:.2f}%')
print('- Detected by self-healing:', report.initial_failures_detected)
print(f'- Recovered@{INITIAL_FAILURE_K}:', primary_recovery.recovered_count)
print(f'- Failed recovery@{INITIAL_FAILURE_K}:', primary_recovery.failed_recovery_count)
if TOP_K in report.recovery_at_cutoff and TOP_K != INITIAL_FAILURE_K:
    recovery_at_top_k = report.recovery_at_cutoff[TOP_K]
    print(f'\nSeparate recovery@{TOP_K}')
    print(f'- Initial failures@{TOP_K}:', recovery_at_top_k.initial_failure_count)
    print(f'- Recovered@{TOP_K}:', recovery_at_top_k.recovered_count)
    print(f'- Failed recovery@{TOP_K}:', recovery_at_top_k.failed_recovery_count)
movement = report.gold_rank_movement
print('\nGold-rank movement')
print('- Improved:', movement.improved_count)
print('- Unchanged:', movement.unchanged_count)
print('- Worsened:', movement.worsened_count)
print('\nSaved metrics:', report.metrics_path)
print('Saved per-example rows:', report.per_example_path)

## 7. Failure examples for inspection

In [ ]:
from src.evaluation import is_failed_recovery_at_cutoff, is_recovered_at_cutoff

def show_examples(label, rows, limit=3):
    selected = list(rows)[:limit]
    print(f'\n{label} ({len(selected)} shown)')
    if not selected:
        print('  None observed in this run.')
    for row in selected:
        print(f'  {row.squad_example_id}: {row.question}')
        print(f'    gold={row.gold_document_id}')
        print(f'    dense/static/initial/final ranks={row.dense_gold_rank}/{row.static_strong_gold_rank}/{row.self_healing_initial_gold_rank}/{row.self_healing_final_gold_rank}')
        print(f'    retries={row.retry_count}, path={" -> ".join(row.graph_path)}')

show_examples(
    'Dense baseline misses',
    (row for row in report.records if row.dense_gold_rank is None),
)
show_examples(
    'Static strong fixes',
    (row for row in report.records if row.dense_gold_rank is None and row.static_strong_gold_rank is not None),
)
show_examples(
    f'Recovered@{INITIAL_FAILURE_K}',
    (row for row in report.records if is_recovered_at_cutoff(row.self_healing_initial_gold_rank, row.self_healing_final_gold_rank, INITIAL_FAILURE_K)),
)
show_examples(
    f'Failed recovery@{INITIAL_FAILURE_K}',
    (row for row in report.records if is_failed_recovery_at_cutoff(row.self_healing_initial_gold_rank, row.self_healing_final_gold_rank, row.retry_count > 0, INITIAL_FAILURE_K)),
)